# Risk Management included Backtest

In [4]:
import os
import re
import time
import faiss
import warnings
import traceback
import numpy as np
import pandas as pd
import quantstats as qs
from tqdm.auto import tqdm
from zoneinfo import ZoneInfo
import google.generativeai as genai
from datetime import datetime, timedelta
# 경고 무시
warnings.filterwarnings('ignore')

# [설정] 경로 및 API 키
BASE_DIR = "/Users/cyberedjs/Desktop/Unity/data"
DATA_DIR = os.path.join(BASE_DIR, "combined")
STRATEGY_NAME = "Volume-Validated Impulse EMA Momentum (VVIEM)"
INDICATOR_NAME = "EMA-Buffered_Momentum_Quality_Score"
RESULT_DIR = f"/Users/cyberedjs/Desktop/Unity/results/backtest_results/{STRATEGY_NAME}"
#PA_TRADE_FILE = f"/Users/cyberedjs/Desktop/Unity/results/pa_trade_results/Momentum/priority/{STRATEGY_NAME}/PA_All_Trades.csv"
STRATEGY_DESC_PATH = f"/Users/cyberedjs/Desktop/Unity/results/pa_trade_results/Momentum/priority/{STRATEGY_NAME}/Strategy_Description.txt"

# 🔥 Google AI Studio API KEY
# optional libs
try:
    from numba import njit, prange, set_num_threads
    set_num_threads(max(1, int(os.environ.get("NUMBA_NUM_THREADS", "3"))))
    USE_NUMBA = True
except Exception:
    USE_NUMBA = False

# Google AI (keeps your original usage)
import google.generativeai as genai
import quantstats as qs

warnings.filterwarnings("ignore")

# ===============================
# CONFIG
# ===============================
INITIAL_CAPITAL = 10_000
PORTFOLIO_RISK_CAP = 0.01       # 1%
PER_TRADE_RISK = 0.001         # 0.25%
MAX_LEVERAGE = 6
MAX_POSITIONS = 10
STOP_ATR = 3
PA_TRADE_FILE = f"/Users/cyberedjs/Desktop/Unity/results/backtest_results/{STRATEGY_NAME}/hold/{INDICATOR_NAME}/Selection_Trades_TOP10.csv"

# ==============================================================================
# 1️⃣ 데이터 로더 및 전처리
# ==============================================================================
def load_market_data(data_dir):
    print("[Data Loader] 데이터 로딩 시작...")
    try:
        data = {}
        for col in ['open', 'high', 'low', 'close', 'volume']:
            path = os.path.join(data_dir, f"{col}_15m.parquet")
            if os.path.exists(path):
                df = pd.read_parquet(path, engine='pyarrow')
                if not isinstance(df.index, pd.DatetimeIndex):
                    df.index = pd.to_datetime(df.index)
                data[col] = df
            else:
                raise FileNotFoundError(f"{path} 파일이 없습니다.")
        print(f"[Data Loader] 로딩 완료. Shape: {data['close'].shape}")
        return data
    except Exception as e:
        print(f"[Error] 데이터 로딩 실패: {e}")
        return None

def load_pa_trades(csv_path):
    print("[Data Loader] PA 거래 내역 로딩 중...")
    if not os.path.exists(csv_path):
        raise FileNotFoundError("PA 거래 내역 파일이 없습니다. Step 1을 먼저 실행하세요.")
    
    df = pd.read_csv(csv_path)
    df['entry_time'] = pd.to_datetime(df['entry_time'])
    df['exit_time'] = pd.to_datetime(df['exit_time'])
    df['anchor'] = pd.to_datetime(df['anchor'])
    return df

# ===============================
# BACKTEST ENGINE
# ===============================
def compute_atr_for_symbol(high, low, close, period=14):
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)

    return tr.rolling(period).mean()

def risk_based_backtest(
    trades: pd.DataFrame,
    highs: pd.DataFrame,
    lows: pd.DataFrame,
    closes: pd.DataFrame,
    atr_period=14,
):
    """
    - 단리 기준: 모든 리스크 계산은 INITIAL_CAPITAL 기준 고정
    - ATR은 trades에 없으므로 OHLC로 entry 시점에서 계산
    - Portfolio risk / per-trade risk / leverage / max positions 모두 반영
    """

    trades = trades.sort_values("entry_time").reset_index(drop=True)

    aum = INITIAL_CAPITAL
    base_capital = INITIAL_CAPITAL  # 🔥 단리 기준
    open_positions = []
    equity_curve = []

    def portfolio_risk():
        return sum(p["risk_amount"] for p in open_positions)

    def portfolio_notional():
        return sum(p["notional"] for p in open_positions)

    # ---------------------------
    # ATR 미리 계산 (symbol별)
    # ---------------------------
    atr_map = {}
    for sym in trades["symbol"].unique():
        atr_map[sym] = compute_atr_for_symbol(
            highs[sym], lows[sym], closes[sym], period=atr_period
        )

    # ---------------------------
    # Build event table
    # ---------------------------
    events = []
    for i, row in trades.iterrows():
        events.append(("entry", row["entry_time"], i))
        events.append(("exit", row["exit_time"], i))

    events = sorted(events, key=lambda x: (x[1], 0 if x[0] == "exit" else 1))
    events_df = pd.DataFrame(events, columns=["type", "time", "idx"])

    # ---------------------------
    # Event loop
    # ---------------------------
    for time, group in events_df.groupby("time"):

        # ===== EXIT FIRST =====
        for _, e in group[group["type"] == "exit"].iterrows():
            idx = e["idx"]
            trade = trades.loc[idx]

            pos = next((p for p in open_positions if p["idx"] == idx), None)
            if pos is None:
                continue

            pnl = (
                trade["direction"]
                * (trade["exit_price"] - pos["entry_price"])
                * pos["position_size"]
            )

            aum += pnl
            open_positions.remove(pos)

            equity_curve.append({
                "time": time,
                "event": "exit",
                "symbol": trade["symbol"],
                "aum": aum,
                "pnl": pnl
            })

        # ===== ENTRY =====
        entry_idxs = group[group["type"] == "entry"]["idx"].tolist()
        if not entry_idxs:
            continue

        candidates = []
        for idx in entry_idxs:
            trade = trades.loc[idx]
            sym = trade["symbol"]
            entry_time = pd.Timestamp(trade["entry_time"])

            rank_list = trade["selected_symbols"]

            atr_series = atr_map.get(sym)
            if atr_series is None or entry_time not in atr_series.index:
                continue

            atr = atr_series.loc[entry_time]
            if np.isnan(atr) or atr <= 0:
                continue

            # 🔥 단리 기준 risk amount
            risk_amount = base_capital * PER_TRADE_RISK

            risk_per_share = STOP_ATR * atr
            position_size = risk_amount / risk_per_share
            notional = position_size * trade["entry_price"]

            # 🔥 [우선순위 계산] 리스트 내 인덱스가 곧 순위 (0부터 시작)
            try:
                # 리스트 안에 내 심볼이 몇 번째에 있는지 찾음
                rank_score = rank_list.index(sym) 
            except (ValueError, AttributeError):
                # 리스트에 없거나 에러 발생 시 최하위 배정
                rank_score = 999 

            candidates.append({
                "idx": idx, 
                "symbol": sym,
                "entry_price": trade["entry_price"], 
                "notional": notional,
                "risk_amount": risk_amount,
                "rank_score": rank_score # 0(1등) < 1(2등) < ...
            })

        # [3] 랭킹 기반 정렬 및 진입 (rank_score 오름차순)
        # 1등(0)이 가장 먼저 리스트 앞쪽으로 옴
        candidates.sort(key=lambda x: x["rank_score"])

        for c in candidates:
            # 포트폴리오 제약조건 (최대 보유 개수 등)
            if len(open_positions) >= MAX_POSITIONS: break
            
            # 리스크 한도 체크
            if portfolio_risk() + c["risk_amount"] > base_capital * PORTFOLIO_RISK_CAP: continue
            if portfolio_notional() + c["notional"] > base_capital * MAX_LEVERAGE: continue
            
            # 랭킹 리스트에 없던 종목(999)은 무시할 수도 있음 (선택사항)
            # if c["rank_score"] == 999: continue

            open_positions.append(c)
            
            equity_curve.append({
                "time": time, 
                "event": "entry", 
                "symbol": c["symbol"],
                "aum": aum, 
                "rank": c["rank_score"] + 1, # 사람이 보기 편하게 1등부터 표기
                "open_positions_count": len(open_positions)
            })

    return pd.DataFrame(equity_curve)

pa_trades_df = load_pa_trades(PA_TRADE_FILE)
market_data = load_market_data(DATA_DIR)
backtest_df = risk_based_backtest(pa_trades_df, market_data["high"], market_data["low"], market_data["close"])
backtest_df

[Data Loader] PA 거래 내역 로딩 중...
[Data Loader] 데이터 로딩 시작...
[Data Loader] 로딩 완료. Shape: (87745, 594)


KeyError: 'position_size'

In [ ]:
df = backtest_df[backtest_df['event'] == 'exit'].copy().set_index('time', drop=True)
yes = df['aum'].resample('D').last().ffill().pct_change()

example_dir = f"/Users/cyberedjs/Desktop/Unity/results/backtest_results/{STRATEGY_NAME}/pass/{INDICATOR_NAME}"
title = f"real_backtest"
html_path = os.path.join(example_dir, f"{title}.html")
qs.reports.html(yes, output=html_path, title=f"Selection Strategy Report")